# Databases with SQLModel

[SQLModel](https://sqlmodel.tiangolo.com/) is a Python library for defining, validating, storing, and querying structured data. It combines ideas from **Pydantic** (typed data models and validation) and **SQLAlchemy** (relational databases and SQL queries), so the same Python class can describe both an object in your code and a table in a database.

## Why use SQLModel for experiment data?

Experiment data quickly becomes difficult to manage in loose CSV files or nested dictionaries. A database gives the data a consistent structure and makes it easier to:

- define required fields and data types for runs, samples, parameters, and results;
- query subsets of experiments without loading every record into memory;
- connect related information, such as experiments, measurements, models, and datasets;
- keep one local SQLite file during development and later move to a larger database system;
- reproduce analyses by storing metadata and results in a predictable schema.

In this notebook, SQLite is used because it requires no database server: the complete database is stored in a local file.

> **Important — restart the kernel every time.** Before starting an example, switching to another example, or running an example again, restart the notebook kernel and then execute that example's cells in order. This prevents old class and table definitions from remaining in memory.
>
> Restarting the kernel resets Python state, but it does **not** delete the SQLite database files. Delete `database.db` or `comicdatabase.db` when you need a completely fresh database.

## Setup

Install SQLModel in the notebook environment. After the installation finishes, **restart the kernel before continuing**, even when the package was already installed.

In [ ]:
!pip install sqlmodel

> **Stop and restart the kernel now.** Then continue with the import cell below and run the remaining cells in order.

## Example 1: A first table — `Hero`

> **Restart the kernel before running this example.** Run all cells in this section from top to bottom.

The first example introduces the basic SQLModel workflow using a `Hero` table. The model defines the structure of each database row:

- `id` is an optional integer and the primary key. The database assigns it when a hero is stored.
- `name` and `secret_name` are required text fields.
- `age` is optional, so a hero can be stored without a known age.

After defining the model, the example creates a SQLite engine, creates the table, opens a `Session`, adds several heroes, and commits the transaction. The final cells use `select()` to query one hero by name and then all heroes whose names begin with `R`.

This same pattern can be adapted for experiment data. For example, a model could represent one experiment run with fields such as `run_id`, `dataset`, `learning_rate`, `seed`, and `score`.

In [ ]:
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

In [ ]:
SQLModel.metadata.clear()
class Hero(SQLModel, table=True, extend_existing=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    secret_name: str
    age: int | None = None

In [ ]:
hero_1 = Hero(name="Deadpond", secret_name="Dive Wilson")
hero_2 = Hero(name="Spider-Boy", secret_name="Pedro Parqueador")
hero_3 = Hero(name="Rusty-Man", secret_name="Tommy Sharp", age=48)
hero_4 = Hero(name="Rosy-Ruepel", secret_name="Rose Sharp", age=32)
engine = create_engine("sqlite:///database.db")


SQLModel.metadata.create_all(engine)

with Session(engine) as session:
    session.add(hero_1)
    session.add(hero_2)
    session.add(hero_3)
    session.add(hero_4)
    session.commit()

### Querying the `Hero` table

A `Session` is the working connection to the database. A `select(Hero)` statement describes which rows should be returned, and `where(...)` adds a filter. Use `.first()` when one result is expected and `.all()` when several results may match.

In [ ]:
with Session(engine) as session:
    statement = select(Hero).where(Hero.name == "Spider-Boy")
    hero = session.exec(statement).first()
    print(hero)

In [ ]:
with Session(engine) as session:
    statement = select(Hero).where(Hero.name.startswith("R"))
    hero = session.exec(statement).all()
    for i in hero:
        print(i)


## Example 2: Linking `Hero` and `Comic`

> **Restart the kernel before running this example.** Do not continue from the in-memory classes used in the first example. The code cell below is self-contained, so run it after restarting the kernel.

Real datasets often contain related entities. In experiment tracking, one experiment might have many runs; similarly, in this example one `Comic` can contain many `Hero` records.

The link is created in two complementary ways:

- `Hero.comic_id` is a **foreign key** that stores the ID of the related row in the `comic` table.
- `Hero.comic` and `Comic.heroes` are `Relationship` attributes that make the connection convenient to use from Python.
- `back_populates` keeps both sides of the relationship synchronized: a comic knows its heroes, and a hero knows its comic.

This is a **one-to-many relationship**: one comic can have many heroes, while each hero belongs to at most one comic. It is sometimes described informally as a “link” example, but it does not use a separate link table. A separate link model is normally needed for a many-to-many relationship.

The example creates a `Comic` named `Marvel`, attaches two hero objects through `heroes=[...]`, and stores the connected objects in `comicdatabase.db`. The following query retrieves `Spider-Boy`; the stored `comic_id` identifies the comic to which that hero belongs.

The same structure is useful for experiment data—for example, an `Experiment` row can have many related `Run` rows, with each run carrying a foreign key back to its experiment.

In [1]:
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

SQLModel.metadata.clear()


class Comic(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str

    heros: list["Hero"] = Relationship(back_populates="comic")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    secret_name: str
    age: int | None = None
    comic_id: int | None = Field(default=None, foreign_key="comic.id")
    comic: Comic | None = Relationship(back_populates="heros")


engine = create_engine("sqlite:///comicdatabase.db")

SQLModel.metadata.create_all(engine)

hero_1 = Hero(name="Deadpond", secret_name="Dive Wilson")
hero_2 = Hero(name="Spider-Boy", secret_name="Pedro Parqueador")

comic_1 = Comic(
    name="Marvel",
    heros=[hero_1, hero_2],
)

with Session(engine) as session:
    session.add(comic_1)
    session.commit()

In [2]:
with Session(engine) as session:
    statement = select(Comic).where(Comic.name == "Marvel")
    comic = session.exec(statement).first()
    print(comic)

id=1 name='Marvel'


In [3]:
with Session(engine) as session:
    statement = select(Comic).where(Comic.name == "Marvel")
    comic = session.exec(statement).first()
    print(comic.heros)

[Hero(id=1, age=None, comic_id=1, name='Deadpond', secret_name='Dive Wilson'), Hero(id=2, age=None, comic_id=1, name='Spider-Boy', secret_name='Pedro Parqueador')]


### Before running the notebook again

Always **restart the kernel first** and execute only the example you want from top to bottom. For a clean database with no previously stored rows, also delete the corresponding SQLite file before running the example.

## Final Exercise: Store Cake Experiments in a Database

So far, your cake experiments have been represented as Python objects and saved to files. This works well for individual experiments, but it becomes difficult to manage once many experiments have been collected.

Extend your experiment framework so that cake experiments can be stored in and retrieved from a relational database.

Use **SQLModel** to define the database structure and **SQLite** as the database system. SQLite stores the complete database in a local file and does not require a separate database server.

### Core Challenge: Store an Experiment

Create a SQLModel table representing a cake experiment.

Your model could contain fields such as:

* experiment ID;
* cake name;
* observer;
* date and time;
* cake type;
* number of layers;
* rating;
* notes.

Choose appropriate data types and decide which fields are required or optional.

Your solution should:

1. define a database model;
2. create a SQLite database and its tables;
3. create at least two cake experiments;
4. add them to the database;
5. commit the changes;
6. query the experiments from the database.

For example, you should be able to find:

* an experiment by its ID;
* all experiments for a particular cake;
* all experiments performed by a particular observer;
* experiments with a rating above a chosen value.

### Advanced Challenge: Store Measurements

Extend the database so that one experiment can contain several measurements.

For example, a cake experiment might have measurements for:

* temperature;
* height;
* moisture;
* mass;
* sweetness;
* structural stability.

Consider using two related tables:

```text
Experiment
    └── Measurement
```

One experiment can have many measurements, while each measurement belongs to one experiment.

Use:

* a primary key to identify each row;
* a foreign key to connect a measurement to its experiment;
* SQLModel relationships to access the connected objects from Python.

NumPy arrays cannot normally be stored directly in a simple database column. Decide how your numerical data should be represented.

Possible approaches include:

* storing each measured value as a separate database row;

Think about which approach makes the data easiest to query and maintain.

### Professional Challenge: Store Image Information

Extend your database design so that an experiment can contain one or more images.

You could introduce an additional table:

```text
Experiment
    ├── Measurement
    └── ImageRecord
```

An image record could contain:

* an image ID;
* the related experiment ID;
* a relative file path;
* a description;
* image width and height;
* number of channels;
* pixel data type;
* acquisition time;
* camera or instrument information.

In most cases, the image itself should remain in an external image file. The database stores the path and the metadata required to understand the image.

Make sure the relationship between the experiment and its images can be reconstructed when the data is queried.

### Test Your Database

Write an automated test that verifies the database workflow.

The test should:

1. create a temporary SQLite database;
2. create the required tables;
3. insert a known cake experiment;
4. query the experiment again;
5. check that the retrieved values match the original values.

Advanced and professional solutions should also test that related measurements or images can be retrieved correctly.

Do not let the test use your normal workshop database. Each test should use a new temporary database so that it remains independent and repeatable.

## Expected Outcome

By the end of the exercise, your framework should be able to:

* create a database;
* store cake experiments;
* retrieve experiments with queries;
* represent relationships between experiments and their data;
* verify the database workflow with an automated test.

Your final framework should now support the complete experimental workflow:

```text
Define → Observe → Store → Load → Query → Test
```

**A cake that is not in the database may never have been observed.**
